# 04. Визуализация графа связей

> **Краткое резюме для руководителя**: Этот ноутбук создаёт интерактивные HTML-карты связей. Компании отображаются как узлы разного цвета и размера, транзакции — как линии между ними. Красные узлы — потенциальные shell-компании. Размер узла отражает его важность (PageRank). Можно визуально оценить структуру бизнес-группы, найти ключевые компании и подозрительные цепочки.

**User Story 4**: Интерактивный граф для демонстрации и анализа.

**Результат**:
- Интерактивный HTML-граф с цветовой кодировкой
- Сводная таблица по кластерам
- Профили ключевых узлов
- Визуализация отдельных кластеров

**Предпосылки**: Запустите `03_graph_analysis.ipynb`.

---

### Легенда визуализации

**Узлы (цвет по типу)**:
- Синий — юридическое лицо (company)
- Зелёный — физическое лицо (individual)
- Оранжевый — ИП (sole_proprietor)
- Красноватый — shell-компания (независимо от типа)

**Узлы (размер)**: пропорционален PageRank — чем важнее узел, тем крупнее

**Рёбра (цвет по типу)**:
- Серый — транзакция (толщина ~ edge_score)
- Красный — доверенность
- Зелёный — зарплатный проект
- Фиолетовый — общие сотрудники

---

In [ ]:
import sys, os, pickle
sys.path.insert(0, '..')

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(levelname)s: %(message)s')

import pandas as pd
from src import config
from src.viz import (
    create_graph_visualization,
    create_cluster_visualization,
    display_summary_table,
    display_node_profile,
)

## Загрузка данных

In [ ]:
# Загружаем анализированный граф и метрики (локальная ФС)
with open(config.OUTPUT_FILTERED_GRAPH_PICKLE, 'rb') as f:
    G = pickle.load(f)

metrics_df = pd.read_parquet(config.OUTPUT_GRAPH_METRICS)
metrics_df = metrics_df.set_index('client_uk') if 'client_uk' in metrics_df.columns else metrics_df

cluster_summary = pd.DataFrame()
if os.path.exists(config.OUTPUT_CLUSTERS):
    cluster_summary = pd.read_parquet(config.OUTPUT_CLUSTERS)

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print(f'Metrics: {len(metrics_df)} rows')
print(f'Clusters: {len(cluster_summary)} rows')

## Полный граф

Интерактивный HTML-граф всех узлов и связей. В большом графе (>500 узлов) рекомендуется использовать визуализацию отдельных кластеров (ниже) для читаемости.

**Как читать**: наведите курсор на узел — появится всплывающая подсказка с именем, типом, ключевыми метриками. Перетаскивайте узлы для перегруппировки.

In [ ]:
output_html = os.path.join(config.DATA_DIR, 'graph.html')
net = create_graph_visualization(G, output_file=output_html)
net.show(output_html)

## Сводная таблица по кластерам

Таблица ранжирована по внутреннему обороту. Обратите внимание на:
- **Кластеры с shell_count > 0** — содержат потенциальные shell-компании
- **Кластеры с has_cycles = True** — обнаружены циклические платежи
- **external_counterparty_count** — много внешних контрагентов = активная торговля с внешним миром

In [ ]:
display_summary_table(cluster_summary)

## Визуализация топ-кластеров

Отдельная интерактивная карта для каждого из трёх крупнейших кластеров по обороту. Позволяет детально рассмотреть внутреннюю структуру группы: кто является центральным узлом, какие связи наиболее сильные, есть ли подозрительные элементы.

In [ ]:
# Визуализация топ-3 кластеров по обороту
if len(cluster_summary) > 0:
    top_clusters = cluster_summary.nlargest(3, 'total_internal_turnover')
    
    for _, row in top_clusters.iterrows():
        cid = int(row['cluster_id'])
        print(f"\n{'='*50}")
        print(f"Cluster {cid}: {row['lead_company_name']}")
        print(f"Members: {int(row['member_count'])}, Turnover: {row['total_internal_turnover']}")
        print('='*50)
        
        cluster_html = os.path.join(config.DATA_DIR, f'cluster_{cid}.html')
        cluster_net = create_cluster_visualization(G, cid, output_file=cluster_html)
        if cluster_net:
            cluster_net.show(cluster_html)
else:
    print('No clusters to visualize.')

## Профиль seed-компании

Детальный профиль компании-«зерна» (hop_distance=0), вокруг которой строился граф. Показывает все метрики, роль, кластер, контрагентов.

In [ ]:
# Найти seed-компанию (hop_distance=0)
seed_nodes = [n for n, d in G.nodes(data=True) if d.get('hop_distance', -1) == 0]

if seed_nodes:
    for seed in seed_nodes:
        print(f'Seed company profile: {G.nodes[seed].get("name", seed)}')
        display_node_profile(G, seed, metrics_df)
else:
    print('No seed node found (hop_distance=0). Enter client_uk manually:')
    # display_node_profile(G, YOUR_CLIENT_UK_HERE, metrics_df)

## Профили shell-компаний

Детальные профили компаний с shell_score выше порога. Для каждой показываются метрики и визуальные связи. Используйте для быстрой оценки: действительно ли компания подозрительна или это ложное срабатывание.

In [ ]:
if 'shell_score' in metrics_df.columns:
    shells = metrics_df[metrics_df['shell_score'] >= config.SHELL_SCORE_THRESHOLD]
    shells = shells.sort_values('shell_score', ascending=False)
    
    if len(shells) > 0:
        print(f'Shell companies: {len(shells)}')
        for node_id in shells.index[:5]:  # Top 5
            display_node_profile(G, node_id, metrics_df)
    else:
        print('No shell companies flagged.')
else:
    print('Shell scores not computed. Run 03_graph_analysis.ipynb first.')

---

## Глоссарий терминов

| Термин | Описание |
|--------|----------|
| **PageRank** | Метрика важности: размер узла на визуализации пропорционален PageRank |
| **Edge Score** | Комплексная оценка связи (0–1): толщина ребра пропорциональна score |
| **Shell-компания** | Узел красного цвета; shell_score ≥ порога |
| **Cluster ID** | Номер группы (сообщества), найденной алгоритмом Leiden |
| **Hop distance** | Расстояние от seed-компании: 0 = seed, 1 = прямой контрагент, 2 = контрагент контрагента |
| **Reciprocal edge** | Двусторонняя связь: есть платежи и A→B, и B→A |

### Как использовать визуализацию

1. **Обзор**: откройте полный граф, найдите крупные узлы (высокий PageRank) — это ключевые компании
2. **Кластеры**: откройте интересующий кластер, проследите внутренние связи
3. **Shell-компании**: проверьте красные узлы — действительно ли они подозрительны
4. **Циклы**: найдите замкнутые цепочки — это потенциальные схемы

---

**Анализ завершён.** Интерактивные HTML-файлы сохранены в `data/`.

Для расширенного анализа см. `05_industry_analysis.ipynb` (отраслевой анализ) и `06_behavioral_segmentation.ipynb` (поведенческая сегментация).